# Extract data for scenicplus

In [1]:
.libPaths(c("/usr/lib/R/library/", "/usr/local/lib/R/site-library/"))

In [2]:
suppressPackageStartupMessages({
    library(Seurat)
    library(SeuratDisk)
    library(repr)
    library(patchwork)
    library(ggplot2)
    library(Signac)
    library(tidyverse)
    library(GenomicRanges)
    library(Matrix)
    library(ggrepel)
    library(fs)
    library(pheatmap)
    library(gridExtra)
    library(RColorBrewer)
    library(data.table)
    library(ComplexHeatmap)
})
options(future.globals.maxSize = Inf)
options(Seurat.object.assay.version = "v5")
options(ggrepel.max.overlaps = Inf)

In [4]:
obj <- readRDS("signac_mgc_combined_peaks_GRN.rds")
obj

An object of class Seurat 
291093 features across 40553 samples within 2 assays 
Active assay: RNA (21808 features, 0 variable features)
 2 layers present: counts, data
 1 other assay present: peaks
 2 dimensional reductions calculated: lsi, umap

In [5]:
## Write MM
writeMM(LayerData(obj, layer = "counts", assay = "peaks"), "./micro-atac.mtx")

NULL

In [6]:
DefaultAssay(obj) <- "peaks"
write.table(
    rownames(obj), 
    "./micro-peaks.tsv",
    col.names = FALSE, 
    quote = FALSE, 
    sep = "\t", 
    row.names = FALSE
)

# Export ATAC to anndata

In [1]:
import scanpy as sc
import pandas as pd

In [2]:
import os

In [3]:
from pathlib import Path

In [4]:
atac_counts = sc.read_mtx(f"micro-atac.mtx")

In [5]:
atac_counts = atac_counts.T.copy()

In [6]:
cell_meta = pd.read_csv(f"micro-meta.tsv", header=0, index_col=0, sep="\t")
region_names = pd.read_csv(f"micro-peaks.tsv", header=None, index_col=0, sep="\t")
region_names.index.name = None

In [7]:
atac_counts.obs = cell_meta
atac_counts.var = region_names

In [8]:
atac_counts

AnnData object with n_obs × n_vars = 40553 × 269285
    obs: 'library_id', 'index', 'orig.ident.x', 'nCount_RNA', 'nFeature_RNA', 'scrubDoublets', 'doublet_scores_obs', 'cell_barcode', 'qc_pass', 'percent.mt', 'tsse', 'frac_dup', 'n_fragment', 'frac_mito', 'doublet_score', 'leverage.score', 'seurat_clusters', 'cluster_full', 'cluster_full.score', 'predicted_allen_class', 'predicted_allen_subclass', 'leverage.score.1', 'subclass', 'neurotransmitter', 'class', 'Senescence_score1', 'group', 'nCount_RNA.trans', 'nFeature_RNA.trans', 'hAPP', 'hPSEN1', 'final_cluster', 'final_celltype', 'final_cluster_color', 'final_celltype_color', 'final_group', 'mouse_id', 'genotype', 'sex', 'age', 'intervention', 'Interferon_related_score1', 'MHC_class_I_score1', 'Activation_Disease_association_score1'

In [9]:
atac_counts.write(f"micro-atac_matrix.h5ad")